# 02 — HNSW Indexing: Como funciona por dentro

**HNSW** (Hierarchical Navigable Small World) e o algoritmo de indexacao mais popular para busca
aproximada de vizinhos em bancos de dados vetoriais.

## Por que HNSW?

| Abordagem | Complexidade | Pros | Contras |
|-----------|-------------|------|---------|
| Brute Force | O(N*D) | Recall 100% | Escala pessimamente |
| HNSW | O(log N) | Excelente recall/velocidade | RAM extra |
| IVF | O(K + sqrt(N)) | Menos RAM | Precisa treino |
| LSH | O(D) | Ultra-rapido | Recall mais baixo |

## Como HNSW funciona

```
Camada 2 (poucos nos, atalhos longos):    A ──── B ──── C
Camada 1 (nos medios, atalhos medios):    A──B──C──D──E──F
Camada 0 (todos os nos, grafo denso):     A-B-C-D-E-F-G-H-I-J...
```

Busca: comecar no topo → usar atalhos para chegar perto → refinar na camada 0

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    HnswConfigDiff, OptimizersConfigDiff
)
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

client = QdrantClient(host='localhost', port=6333)
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Pronto!')

## 2.1 Os Parametros HNSW

### `m` — Numero de conexoes por no
- Controla a **densidade do grafo**
- Valores tipicos: 4 a 64
- Default Qdrant: 16
- **↑ m = melhor recall + mais RAM + indexacao mais lenta**

### `ef_construct` — Candidatos durante construcao
- Candidatos avaliados ao inserir cada no
- Valores tipicos: 50 a 500
- Default Qdrant: 100
- **↑ ef_construct = melhor qualidade do indice + indexacao mais lenta**

### `ef` — Candidatos durante busca (query time)
- Candidatos avaliados em cada busca
- Default: igual a `m * 2`
- **↑ ef = melhor recall + busca mais lenta**

In [ ]:
# Gerar dataset sintetico para experimentar
import random
random.seed(42)
np.random.seed(42)

n_docs = 2000
temas = ['machine learning', 'banco de dados', 'cloud computing', 'seguranca', 
         'frontend', 'backend', 'DevOps', 'data science']

docs_sinteticos = [
    f'{random.choice(temas)}: documento numero {i} sobre {random.choice(["fundamentos", "avancado", "pratico"])}'
    for i in range(n_docs)
]

print(f'Criando embeddings para {n_docs} documentos...')
t0 = time.time()
all_embs = model.encode(docs_sinteticos, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
print(f'Tempo: {time.time()-t0:.1f}s | Shape: {all_embs.shape}')

In [ ]:
# Comparar diferentes configuracoes HNSW
configs_hnsw = [
    {'m': 4,  'ef_construct': 50,  'nome': 'Leve (m=4, ef=50)'},
    {'m': 16, 'ef_construct': 100, 'nome': 'Default (m=16, ef=100)'},
    {'m': 32, 'ef_construct': 200, 'nome': 'Qualidade (m=32, ef=200)'},
]

resultados_hnsw = []

for config in configs_hnsw:
    nome_col = f'hnsw_m{config["m"]}_ef{config["ef_construct"]}'
    
    # Criar collection com config especifica
    if client.collection_exists(nome_col):
        client.delete_collection(nome_col)
    client.create_collection(
        collection_name=nome_col,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        hnsw_config=HnswConfigDiff(
            m=config['m'],
            ef_construct=config['ef_construct'],
        ),
    )
    
    # Indexar documentos
    points = [
        PointStruct(id=i, vector=all_embs[i].tolist(), payload={'texto': docs_sinteticos[i]})
        for i in range(n_docs)
    ]
    
    t0 = time.time()
    client.upsert(collection_name=nome_col, points=points)
    index_time = time.time() - t0
    
    # Benchmark de busca
    n_queries = 100
    query_indices = np.random.choice(n_docs, n_queries, replace=False)
    
    t0 = time.time()
    for qi in query_indices:
        client.query_points(nome_col, query=all_embs[qi].tolist(), limit=10).points
    search_time = (time.time() - t0) / n_queries * 1000  # ms por query
    
    # Info da collection
    info = client.get_collection(nome_col)
    
    resultados_hnsw.append({
        'Config': config['nome'],
        'm': config['m'],
        'ef_construct': config['ef_construct'],
        'Indexacao (s)': f'{index_time:.2f}',
        'Busca (ms/query)': f'{search_time:.2f}',
    })
    print(f'{config["nome"]}: index={index_time:.2f}s | search={search_time:.2f}ms/query')

df_hnsw = pd.DataFrame(resultados_hnsw)
print('\nTabela Comparativa HNSW:')
print(df_hnsw.to_string(index=False))

In [ ]:
# Visualizacao: tradeoff de configuracoes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ms = [c['m'] for c in configs_hnsw]
labels = [c['nome'] for c in configs_hnsw]
cores = ['#3498db', '#2ecc71', '#e74c3c']

# Memoria estimada (proporcional a m)
mem_estimada = [m * 4 for m in ms]  # bytes adicionais por ponto, aprox
axes[0].bar(labels, mem_estimada, color=cores)
axes[0].set_title('Memoria Estimada do Indice HNSW\n(proporcional a m)', fontweight='bold')
axes[0].set_ylabel('Memoria relativa')
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=20, ha='right', fontsize=9)

# Relacao recall estimado x velocidade (conceitual)
recall_estimado = [0.85, 0.95, 0.99]  # estimados
busca_ms = [float(r['Busca (ms/query)']) for r in resultados_hnsw]

for i, (rec, ms_val, label) in enumerate(zip(recall_estimado, busca_ms, labels)):
    axes[1].scatter(ms_val, rec, s=200, color=cores[i], label=label, zorder=3)
    axes[1].annotate(label, (ms_val, rec), xytext=(3, 3), textcoords='offset points', fontsize=8)

axes[1].set_title('Trade-off Recall vs Velocidade de Busca', fontweight='bold')
axes[1].set_xlabel('Tempo de Busca (ms/query)')
axes[1].set_ylabel('Recall Estimado')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impacto dos Parametros HNSW', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Guia de Configuracao

| Cenario | m | ef_construct | ef | RAM extra |
|---------|---|-------------|-----|----------|
| Prototipo / dev | 8 | 50 | 64 | Minima |
| Producao balanceado | 16 | 100 | 128 | Moderada |
| Alta qualidade | 32 | 200 | 256 | Alta |
| Recall maximo | 64 | 500 | 512 | Muito alta |

**Regra pratica:**
- Para colecoes < 100K docs: defaults sao perfeitos
- Para > 1M docs: aumente m e ef_construct gradualmente
- Sempre meca o recall antes de ir para producao!

## Proximo
- [03 — Quantizacao](03_quantization.html)